In [2]:
# ============================================================
# HYROX BILBAO 2026 — LIMPIEZA Y TRANSFORMACIÓN
# Input:  hyrox_bilbao_2026_raw.csv
# Output: hyrox_bilbao_2026_clean.csv
# ============================================================

from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/hyrox'

print("✅ Drive montado")
print(f"\nArchivos disponibles:")
for a in sorted(os.listdir(RUTA)):
    print(f"  {a}")

df = pd.read_csv(f'{RUTA}/hyrox_bilbao_2026_raw.csv')
df_clean = df.copy()
print(f"\nRaw cargado: {len(df_clean)} registros")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado

Archivos disponibles:
  hyox_page0.html
  hyrox_bilbao_2026_clean.csv
  hyrox_bilbao_2026_raw.csv
  hyrox_bilbao_2026_splits.csv
  hyrox_page1.html
  hyrox_page2.html
  hyrox_page3.html
  hyrox_page4.html
  hyrox_page5.html
  hyrox_page6.html
  hyrox_page7.html
  hyrox_page8.html

Raw cargado: 861 registros


In [3]:
# PASO 1 — Eliminar columna auxiliar de paginación
df_clean = df_clean.drop(columns=['pagina'])
print("✅ Columna 'pagina' eliminada")

✅ Columna 'pagina' eliminada


In [4]:
# PASO 2 — Convertir posiciones a entero
df_clean['pos_general'] = pd.to_numeric(df_clean['pos_general'], errors='coerce').astype('Int64')
df_clean['pos_categoria'] = pd.to_numeric(df_clean['pos_categoria'], errors='coerce').astype('Int64')
print("✅ Posiciones convertidas a numérico")
print(df_clean[['pos_general', 'pos_categoria']].dtypes)

✅ Posiciones convertidas a numérico
pos_general      Int64
pos_categoria    Int64
dtype: object


In [5]:
# PASO 3 — Separar los dos rankings entrelazados
# La web mezcla parejas mixtas y Men en la misma tabla
# Las parejas Men puras tienen tiempos coherentes (sub 2h aprox)
# Las parejas mixtas tienen tiempos anómalos (>2h o duplicados de posición)

# Identificar filas duplicadas en pos_general
duplicados = df_clean[df_clean.duplicated(subset='pos_general', keep=False)]
print(f"Filas con pos_general duplicada: {len(duplicados)}")
print(f"Posiciones únicas duplicadas: {duplicados['pos_general'].nunique()}")
print("\nEjemplo de duplicados:")
print(duplicados.head(6)[['pos_general', 'nombres_pareja', 'grupo_edad', 'tiempo_total']])

Filas con pos_general duplicada: 12
Posiciones únicas duplicadas: 6

Ejemplo de duplicados:
   pos_general                                  nombres_pareja grupo_edad  \
0            1                 Álvaro Villegas, Teresa Bartret      25-29   
1            1  JOSE AGUSTIN ALISES GIMENEZ, LUIS GARCIA RUBIO      25-29   
2            2                    Alazne Iturriaga, Pablo Oria      25-29   
3            2    Leonardo Alonso Mora, Sergio López Izquierdo      25-29   
4            3                     Vanessa Lopes, Romain Lopes      40-44   
5            3                Bernardo Branco, Ricardo Fonseca      25-29   

  tiempo_total  
0     01:07:44  
1     00:50:14  
2     01:24:33  
3     00:51:46  
4     01:26:49  
5     00:53:08  


In [6]:
# PASO 4 — Resolver duplicados: quedarse con el menor tiempo por posición
def tiempo_a_segundos(t):
    try:
        h, m, s = t.strip().split(':')
        return int(h)*3600 + int(m)*60 + int(s)
    except:
        return None

df_clean['tiempo_segundos'] = df_clean['tiempo_total'].apply(tiempo_a_segundos)

# Para duplicados, conservar la fila con menor tiempo
df_clean = df_clean.sort_values('tiempo_segundos')
df_clean = df_clean.drop_duplicates(subset='pos_general', keep='first')
df_clean = df_clean.sort_values('pos_general').reset_index(drop=True)

print(f"✅ Registros tras resolver duplicados: {len(df_clean)}")
print(f"Rango posiciones: {df_clean['pos_general'].min()} → {df_clean['pos_general'].max()}")
print("\nVerificación — primeras 10 filas:")
print(df_clean[['pos_general', 'nombres_pareja', 'grupo_edad', 'tiempo_total']].head(10).to_string())

✅ Registros tras resolver duplicados: 855
Rango posiciones: 1 → 855

Verificación — primeras 10 filas:
   pos_general                                  nombres_pareja grupo_edad tiempo_total
0            1  JOSE AGUSTIN ALISES GIMENEZ, LUIS GARCIA RUBIO      25-29     00:50:14
1            2    Leonardo Alonso Mora, Sergio López Izquierdo      25-29     00:51:46
2            3                Bernardo Branco, Ricardo Fonseca      25-29     00:53:08
3            4               Raul Sevillano, Pedro Jose Olmedo      25-29     00:54:02
4            5                  bradley johnson, Tom Cansfield      35-39     00:54:11
5            6                Théo Dos Santos, Enzo Dos Santos      16-24     00:54:34
6            7    Jorge Casao Orna, Carlos Sanjuan Ballesteros      30-34     00:54:40
7            8      Miguel Extremera Vacas, David Peña Anguita      35-39     00:54:48
8            9          Javier Zorrilla Calleja, Michael Fummo      30-34     00:54:57
9           10             

In [7]:
# PASO 5 — Normalizar nombres a Title Case
df_clean['nombres_pareja'] = df_clean['nombres_pareja'].str.title()

# Separar atleta_1 y atleta_2
df_clean[['atleta_1', 'atleta_2']] = df_clean['nombres_pareja'].str.split(',', n=1, expand=True)
df_clean['atleta_1'] = df_clean['atleta_1'].str.strip()
df_clean['atleta_2'] = df_clean['atleta_2'].str.strip()

print("✅ Nombres normalizados y separados")
print(df_clean[['atleta_1', 'atleta_2']].head(5))

✅ Nombres normalizados y separados
                      atleta_1                atleta_2
0  Jose Agustin Alises Gimenez       Luis Garcia Rubio
1         Leonardo Alonso Mora  Sergio López Izquierdo
2              Bernardo Branco         Ricardo Fonseca
3               Raul Sevillano       Pedro Jose Olmedo
4              Bradley Johnson           Tom Cansfield


In [8]:
# PASO 6 — Añadir columnas calculadas y metadatos
df_clean['tiempo_minutos'] = (df_clean['tiempo_segundos'] / 60).round(2)

df_clean['evento']    = 'Bilbao'
df_clean['año']       = 2026
df_clean['modalidad'] = 'Doubles Men'

print("✅ Columnas calculadas y metadatos añadidos")
print(f"\nRango de tiempos:")
print(f"  Mínimo: {df_clean['tiempo_total'].min()}")
print(f"  Máximo: {df_clean['tiempo_total'].max()}")
print(f"  Media:  {df_clean['tiempo_minutos'].mean():.1f} minutos")

✅ Columnas calculadas y metadatos añadidos

Rango de tiempos:
  Mínimo: 00:50:14
  Máximo: 01:52:14
  Media:  70.1 minutos


In [9]:
# PASO 7 — Guardar CSV procesado
columnas_finales = [
    'pos_general', 'pos_categoria', 'atleta_1', 'atleta_2',
    'nombres_pareja', 'grupo_edad', 'tiempo_total',
    'tiempo_segundos', 'tiempo_minutos', 'evento', 'año', 'modalidad'
]

df_clean = df_clean[columnas_finales]
df_clean.to_csv('hyrox_bilbao_2026_clean.csv', index=False)

print("✅ Guardado: hyrox_bilbao_2026_clean.csv")
print(f"   {len(df_clean)} registros | {df_clean.shape[1]} columnas")
print(f"\nColumnas: {list(df_clean.columns)}")

✅ Guardado: hyrox_bilbao_2026_clean.csv
   855 registros | 12 columnas

Columnas: ['pos_general', 'pos_categoria', 'atleta_1', 'atleta_2', 'nombres_pareja', 'grupo_edad', 'tiempo_total', 'tiempo_segundos', 'tiempo_minutos', 'evento', 'año', 'modalidad']


In [10]:
# CELDA — Cargar splits y hacer join con el dataset clean
df_clean = pd.read_csv(f'{RUTA}/hyrox_bilbao_2026_clean.csv')
df_splits = pd.read_csv(f'{RUTA}/hyrox_bilbao_2026_splits.csv')

print(f"Clean: {len(df_clean)} registros")
print(f"Splits: {len(df_splits)} registros")

Clean: 855 registros
Splits: 861 registros


In [11]:
# Join por nombres_pareja
df_master = df_clean.merge(df_splits, on='nombres_pareja', how='left')

print(f"Master: {len(df_master)} registros")
print(f"Columnas: {df_master.shape[1]}")
print(f"\nNulos tras el join:")
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])

Master: 855 registros
Columnas: 31

Nulos tras el join:
running_1                160
1000m_skierg             160
running_2                160
50m_sled_push            160
running_3                160
50m_sled_pull            160
running_4                160
80m_burpee_broad_jump    160
running_5                160
1000m_row                160
running_6                160
200m_farmers_carry       160
running_7                160
100m_sandbag_lunges      160
running_8                160
wall_balls               160
roxzone_time             160
run_total                160
best_run_lap             160
dtype: int64


In [12]:
# Guardar dataset maestro
df_master.to_csv(f'{RUTA}/hyrox_bilbao_2026_master.csv', index=False)
print("✅ Guardado: hyrox_bilbao_2026_master.csv")
print(f"   {len(df_master)} registros | {df_master.shape[1]} columnas")
print(f"\nColumnas finales:")
for c in df_master.columns:
    print(f"  {c}")

✅ Guardado: hyrox_bilbao_2026_master.csv
   855 registros | 31 columnas

Columnas finales:
  pos_general
  pos_categoria
  atleta_1
  atleta_2
  nombres_pareja
  grupo_edad
  tiempo_total
  tiempo_segundos
  tiempo_minutos
  evento
  año
  modalidad
  running_1
  1000m_skierg
  running_2
  50m_sled_push
  running_3
  50m_sled_pull
  running_4
  80m_burpee_broad_jump
  running_5
  1000m_row
  running_6
  200m_farmers_carry
  running_7
  100m_sandbag_lunges
  running_8
  wall_balls
  roxzone_time
  run_total
  best_run_lap


In [13]:
# Ver qué nombres no hacen match
nombres_clean = set(df_clean['nombres_pareja'])
nombres_splits = set(df_splits['nombres_pareja'])

sin_match = nombres_clean - nombres_splits
print(f"Parejas en clean sin match en splits: {len(sin_match)}")
print("\nEjemplos:")
for n in list(sin_match)[:10]:
    print(f"  CLEAN:  '{n}'")
    # Buscar el más parecido en splits
    candidatos = [s for s in nombres_splits if s.split(',')[0].strip().lower() in n.lower()]
    if candidatos:
        print(f"  SPLITS: '{candidatos[0]}'")
    print()

Parejas en clean sin match en splits: 160

Ejemplos:
  CLEAN:  'Adrián Pérez González, Armando Silva Rojas'
  SPLITS: 'ADRIÁN PÉREZ GONZÁLEZ, ARMANDO SILVA ROJAS'

  CLEAN:  'Marc Pereira Clemente, François Macias'
  SPLITS: 'MARC PEREIRA CLEMENTE, François Macias'

  CLEAN:  'Alex Fernandez Estefanía, Pablo Garcia Monco'
  SPLITS: 'Pablo Garcia, José Ramon Llama'

  CLEAN:  'Arturo Llames Garcia, Adriá Herrera Juan'
  SPLITS: 'ARTURO LLAMES GARCIA, ADRIÁ HERRERA JUAN'

  CLEAN:  'Jules Rageot-Bologna, Marius Savy'
  SPLITS: 'Jules Rageot-Bologna, Marius SAVY'

  CLEAN:  'Fabrice Kavadioko, Dimitri Niverly'
  SPLITS: 'Fabrice KAVADIOKO, Dimitri Niverly'

  CLEAN:  'Yann Personnic, David Gomes'
  SPLITS: 'Yann Personnic, David GOMES'

  CLEAN:  'Aitor Chousa, Unai Martínez De Lizarrondo'
  SPLITS: 'Aitor Chousa, Unai Martínez de Lizarrondo'

  CLEAN:  'Héctor Gisbert Bernal, Carlos Mazon Abril'
  SPLITS: 'Héctor Gisbert Bernal, Carlos Mazon abril'

  CLEAN:  'Asis Barrenechea, Bosco Unz

In [14]:
# Join insensible a mayúsculas
df_clean['_key'] = df_clean['nombres_pareja'].str.lower().str.strip()
df_splits['_key'] = df_splits['nombres_pareja'].str.lower().str.strip()

df_master = df_clean.merge(
    df_splits.drop(columns='nombres_pareja'),
    on='_key',
    how='left'
)

df_master = df_master.drop(columns='_key')

print(f"Master: {len(df_master)} registros")
print(f"Columnas: {df_master.shape[1]}")
print(f"\nNulos tras el join:")
nulos = df_master.isnull().sum()
nulos = nulos[nulos > 0]
print(nulos if len(nulos) > 0 else "✅ Sin nulos")

Master: 855 registros
Columnas: 31

Nulos tras el join:
✅ Sin nulos


In [15]:
df_master.to_csv(f'{RUTA}/hyrox_bilbao_2026_master.csv', index=False)
print("✅ Guardado: hyrox_bilbao_2026_master.csv")
print(f"   {len(df_master)} registros | {df_master.shape[1]} columnas")
print(f"\nColumnas finales:")
for c in df_master.columns:
    print(f"  {c}")

✅ Guardado: hyrox_bilbao_2026_master.csv
   855 registros | 31 columnas

Columnas finales:
  pos_general
  pos_categoria
  atleta_1
  atleta_2
  nombres_pareja
  grupo_edad
  tiempo_total
  tiempo_segundos
  tiempo_minutos
  evento
  año
  modalidad
  running_1
  1000m_skierg
  running_2
  50m_sled_push
  running_3
  50m_sled_pull
  running_4
  80m_burpee_broad_jump
  running_5
  1000m_row
  running_6
  200m_farmers_carry
  running_7
  100m_sandbag_lunges
  running_8
  wall_balls
  roxzone_time
  run_total
  best_run_lap
